# Notebook 03-CIC — Calibration on CIC-IDS2017

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection 
**Author:** Md Anas Biswas, University of Portsmouth 
**Dataset:** CIC-IDS2017 (200K subsample)

## Methodology

Same as Notebook 03 v2 (NSL-KDD): split test set 50/50 into calibration + eval halves. Fit isotonic regression on the calibration half, evaluate ECE/Brier on the eval half. Reuse the same calibrator choice per model (isotonic globally for trees, isotonic per-class for DNN; we'll let the data decide here too).

## Output

```
calibrators/cic_ids2017/
├── idx_calib.npy / idx_eval.npy
├── calibrated_probabilities/<name>_calibrated.npy
└── *.pkl  (fitted calibrators per model)
results/tables/cic_calibration.csv
results/figures/cic_reliability_diagrams.png
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import json, pickle
from pathlib import Path
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

PROCESSED = Path(REPO) / 'data' / 'processed' / 'cic_ids2017'
MODELS_DIR = Path(REPO) / 'models' / 'cic_ids2017'
PREDS_DIR = MODELS_DIR / 'predictions'
CALIB_DIR = Path(REPO) / 'calibrators' / 'cic_ids2017'
CALIB_PROBS_DIR = CALIB_DIR / 'calibrated_probabilities'
CALIB_DIR.mkdir(parents=True, exist_ok=True)
CALIB_PROBS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path(REPO) / 'results' / 'figures'
TABLES_DIR = Path(REPO) / 'results' / 'tables'

y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_BINARY = ['Normal', 'Attack']
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'Test set: {len(y_test_b):,} samples')
print(f'5-class: {np.bincount(y_test_5)}')

---
## Step 1 — Split test 50/50 (stratified by 5-class)

Use 5-class for stratification so calibration set sees all attack types.

In [ ]:
from sklearn.model_selection import train_test_split

all_idx = np.arange(len(y_test_5))
idx_calib, idx_eval = train_test_split(
    all_idx, test_size=0.50, stratify=y_test_5, random_state=SEED
)

np.save(CALIB_DIR / 'idx_calib.npy', idx_calib)
np.save(CALIB_DIR / 'idx_eval.npy', idx_eval)

print(f'Calibration half: {len(idx_calib):,}')
print(f'Eval half:        {len(idx_eval):,}')
print(f'5-class in calib: {np.bincount(y_test_5[idx_calib])}')
print(f'5-class in eval:  {np.bincount(y_test_5[idx_eval])}')

---
## Step 2 — ECE / Brier helpers

In [ ]:
def expected_calibration_error(y_true, y_proba, n_bins=15):
    '''Top-label ECE.'''
    confidences = y_proba.max(axis=1)
    predictions = y_proba.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        in_bin = (confidences > bins[i]) & (confidences <= bins[i+1])
        bs = in_bin.sum()
        if bs > 0:
            bin_acc = accuracies[in_bin].mean()
            bin_conf = confidences[in_bin].mean()
            ece += (bs / n) * abs(bin_acc - bin_conf)
    return float(ece)

def brier_multiclass(y_true, y_proba):
    '''Multiclass Brier: mean over samples of sum_c (y_c - p_c)^2.'''
    n_classes = y_proba.shape[1]
    y_onehot = np.zeros_like(y_proba)
    y_onehot[np.arange(len(y_true)), y_true] = 1
    return float(((y_proba - y_onehot) ** 2).sum(axis=1).mean())

---
## Step 3 — Fit isotonic calibrators (per-class for multi-class)

In [ ]:
def fit_isotonic_perclass(proba_calib, y_calib):
    '''Fit one isotonic regression per class. Returns list of calibrators.'''
    n_classes = proba_calib.shape[1]
    calibrators = []
    for c in range(n_classes):
        y_c = (y_calib == c).astype(int)
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        iso.fit(proba_calib[:, c], y_c)
        calibrators.append(iso)
    return calibrators

def apply_isotonic_perclass(calibrators, proba):
    '''Apply per-class isotonic and renormalise.'''
    out = np.zeros_like(proba)
    for c, iso in enumerate(calibrators):
        out[:, c] = iso.predict(proba[:, c])
    # Renormalise to sum to 1
    row_sums = out.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return out / row_sums

CANONICAL = {
    'rf_binary_cw':      'binary',
    'xgb_binary_cw':     'binary',
    'dnn_binary_cw':     'binary',
    'rf_5class_smote':   '5class',
    'xgb_5class_smote':  '5class',
    'dnn_5class_smote':  '5class',
}

results = []
for name, target in CANONICAL.items():
    print(f'\n--- {name} ---')
    y_test = y_test_b if target == 'binary' else y_test_5
    proba = np.load(PREDS_DIR / f'{name}_proba.npy')
    
    proba_calib = proba[idx_calib]
    proba_eval  = proba[idx_eval]
    y_calib = y_test[idx_calib]
    y_eval  = y_test[idx_eval]
    
    # Uncalibrated baseline on eval
    ece_uncal = expected_calibration_error(y_eval, proba_eval)
    brier_uncal = brier_multiclass(y_eval, proba_eval)
    print(f'  Uncalibrated: ECE={ece_uncal:.4f}, Brier={brier_uncal:.4f}')
    
    # Fit per-class isotonic
    calibrators = fit_isotonic_perclass(proba_calib, y_calib)
    proba_cal_eval = apply_isotonic_perclass(calibrators, proba_eval)
    
    ece_cal = expected_calibration_error(y_eval, proba_cal_eval)
    brier_cal = brier_multiclass(y_eval, proba_cal_eval)
    print(f'  Calibrated  : ECE={ece_cal:.4f}, Brier={brier_cal:.4f}  ({100*(ece_uncal-ece_cal)/max(ece_uncal,1e-9):+.1f}% ECE reduction)')
    
    # Apply calibrator to full test set, save
    proba_cal_full = apply_isotonic_perclass(calibrators, proba)
    np.save(CALIB_PROBS_DIR / f'{name}_calibrated.npy', proba_cal_full[idx_eval].astype(np.float32))
    with open(CALIB_DIR / f'{name}_calibrator.pkl', 'wb') as f:
        pickle.dump({'method': 'isotonic_perclass', 'calibrators': calibrators}, f)
    
    results.append({
        'Model': name, 'Target': target,
        'ECE_before': ece_uncal, 'ECE_after': ece_cal,
        'ECE_reduction_pct': 100 * (ece_uncal - ece_cal) / max(ece_uncal, 1e-9),
        'Brier_before': brier_uncal, 'Brier_after': brier_cal,
    })

df = pd.DataFrame(results)
df.to_csv(TABLES_DIR / 'cic_calibration.csv', index=False)
print('\n' + '='*90)
print('CIC-IDS2017 CALIBRATION RESULTS')
print('='*90)
print(df.to_string(index=False, float_format='%.4f'))
print('='*90)

---
## Step 4 — Reliability diagrams

In [ ]:
def reliability_data(y_true, y_proba, n_bins=15):
    confidences = y_proba.max(axis=1)
    predictions = y_proba.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    centres, accs, counts = [], [], []
    for i in range(n_bins):
        in_bin = (confidences > bins[i]) & (confidences <= bins[i+1])
        if in_bin.sum() > 0:
            centres.append((bins[i] + bins[i+1]) / 2)
            accs.append(accuracies[in_bin].mean())
            counts.append(in_bin.sum())
    return np.array(centres), np.array(accs), np.array(counts)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, target) in zip(axes.flatten(), CANONICAL.items()):
    y_test = y_test_b if target == 'binary' else y_test_5
    proba = np.load(PREDS_DIR / f'{name}_proba.npy')
    proba_eval = proba[idx_eval]
    y_eval = y_test[idx_eval]
    proba_cal_eval = np.load(CALIB_PROBS_DIR / f'{name}_calibrated.npy')
    
    c1, a1, _ = reliability_data(y_eval, proba_eval)
    c2, a2, _ = reliability_data(y_eval, proba_cal_eval)
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect')
    ax.plot(c1, a1, 'o-', color='#DD8452', label='Uncalibrated')
    ax.plot(c2, a2, 's-', color='#4C72B0', label='Calibrated')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'cic_reliability_diagrams.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Commit
os.chdir(REPO)
!git add notebooks/03_cic_calibration.ipynb
!git add results/
!git status --short
!git commit -m 'Notebook 03-CIC: calibration on CIC-IDS2017 (per-class isotonic)'
!git push

---
## Summary

Per-class isotonic regression applied. Expected: smaller ECE reductions than NSL-KDD because models are already near-perfect on CIC-IDS2017 (less to calibrate).

**Next:** Notebook 04-CIC (SHAP) and Notebook 05-CIC (stability).
